# Détection de botnets avec les features d'entropie de Lakhina

Ce notebook reprend le pipeline final du projet :
- exploration du dataset CTU-13
- création / rechargement des splits temporels
- sélection du meilleur détecteur
- calibration du seuil
- évaluation finale et sauvegarde des sorties

In [19]:
from copy import deepcopy
from pathlib import Path
import importlib
import json

import pandas as pd
from IPython.display import Image, display

import analysis.statistics as _statistics
import evaluation.metrics as _metrics
import main_detection as _main_detection
import preprocessing.loader as _loader

# Recharge explicitement les modules du projet pour éviter les imports
# obsolètes quand le notebook reste ouvert après des modifications.
_statistics = importlib.reload(_statistics)
_metrics = importlib.reload(_metrics)
_main_detection = importlib.reload(_main_detection)
_loader = importlib.reload(_loader)

compute_entropy_preview = _statistics.compute_entropy_preview
plot_feature_distributions = _statistics.plot_feature_distributions
plot_label_distribution = _statistics.plot_label_distribution
plot_protocol_by_label = _statistics.plot_protocol_by_label
plot_traffic_over_time = _statistics.plot_traffic_over_time
print_summary = _statistics.print_summary

DetectionEvaluator = _metrics.DetectionEvaluator

DETECTION_CONFIG = _main_detection.CONFIG
_plot_calibration_curve = _main_detection._plot_calibration_curve
load_splits = _main_detection.load_splits
print_split_overview = _main_detection.print_split_overview
save_config_and_metrics = _main_detection.save_config_and_metrics
select_best_detector = _main_detection.select_best_detector

DEFAULT_TRAIN_RATIO = _loader.DEFAULT_TRAIN_RATIO
DEFAULT_VAL_RATIO = _loader.DEFAULT_VAL_RATIO
clean_dataframe = _loader.clean_dataframe
load_binetflow = _loader.load_binetflow
save_splits = _loader.save_splits
split_dataset = _loader.split_dataset


ImportError: cannot import name 'DEFAULT_TRAIN_RATIO' from 'preprocessing.loader' (/Users/macbookpro/Documents/IMT/CYBER/supervision/supsys-lakhina-entropy-detection/preprocessing/loader.py)

In [ ]:
CONFIG = deepcopy(DETECTION_CONFIG)
DATASET_PATH = CONFIG['dataset_path']
RESULTS_DIR = Path(CONFIG['results_dir'])
SPLITS_DIR = CONFIG['splits_dir']
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)

def show_saved_figure(filename: str):
    path = RESULTS_DIR / filename
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Figure not found: {path}')

CONFIG

## 1. Exploration du dataset

Cette partie reprend `main_exploration.py` : chargement, nettoyage, résumé et figures d'exploration.

In [ ]:
df_raw = load_binetflow(DATASET_PATH)
df = clean_dataframe(df_raw)
print_summary(df)
df.head()

In [ ]:
plot_label_distribution(df, str(RESULTS_DIR))
show_saved_figure('label_distribution.png')

In [ ]:
plot_traffic_over_time(df, window='5min', save_dir=str(RESULTS_DIR))
show_saved_figure('traffic_over_time.png')

In [ ]:
plot_feature_distributions(df, str(RESULTS_DIR))
show_saved_figure('feature_distributions.png')

In [ ]:
plot_protocol_by_label(df, str(RESULTS_DIR))
show_saved_figure('protocol_by_label.png')

In [ ]:
df_entropy_preview = compute_entropy_preview(
    df,
    window_seconds=CONFIG['window_seconds'],
    save_dir=str(RESULTS_DIR),
)
show_saved_figure('entropy_preview.png')
df_entropy_preview.head()

In [ ]:
df_train, df_val, df_test = split_dataset(
    df,
    train_ratio=DEFAULT_TRAIN_RATIO,
    val_ratio=DEFAULT_VAL_RATIO,
)

split_paths, split_metadata = save_splits(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    splits_dir=SPLITS_DIR,
    source_path=DATASET_PATH,
    train_ratio=DEFAULT_TRAIN_RATIO,
    val_ratio=DEFAULT_VAL_RATIO,
)

pd.Series(
    {
        'train_rows': len(df_train),
        'val_rows': len(df_val),
        'test_rows': len(df_test),
        'metadata_path': str(split_paths['metadata']),
    }
)

## 2. Détection et calibration

Cette partie reprend `main_detection.py` avec la sélection du meilleur modèle sur validation.

In [ ]:
df_train, df_val, df_test = load_splits(CONFIG)
print_split_overview(df_train, df_val, df_test)

In [ ]:
detector, selection_info = select_best_detector(df_train, df_val, CONFIG)

CONFIG['model_name'] = selection_info['model_name']
CONFIG['window_seconds'] = selection_info['window_seconds']
CONFIG['n_components'] = selection_info['n_components']
CONFIG['min_flows'] = selection_info['min_flows']
CONFIG['selected_model'] = {
    'model_name': selection_info['model_name'],
    'window_seconds': selection_info['window_seconds'],
    'n_components': selection_info['n_components'],
    'min_flows': selection_info['min_flows'],
    'threshold': selection_info['threshold'],
    'validation_metrics': selection_info['val_metrics'],
}

pd.Series(
    {
        'selected_model': selection_info['model_name'],
        'window_seconds': selection_info['window_seconds'],
        'n_components': selection_info['n_components'],
        'min_flows': selection_info['min_flows'],
        'threshold': selection_info['threshold'],
        **selection_info['val_metrics'],
    }
)

In [ ]:
optimal_threshold = detector.threshold

if hasattr(detector, '_calibration_data'):
    _plot_calibration_curve(
        detector._calibration_data,
        optimal_threshold,
        save_dir=str(RESULTS_DIR),
    )

show_saved_figure('threshold_calibration.png')

pd.Series(
    {
        'optimal_threshold': optimal_threshold,
        'calibration_metric': CONFIG['calibration_metric'],
        'max_fpr_constraint': CONFIG['calibration_max_fpr'],
    }
)

## 3. Prédiction et évaluation

In [ ]:
results = detector.predict(df_test)

results_path = RESULTS_DIR / 'predictions.parquet'
results.to_parquet(results_path)

print(f'Prédictions sauvegardées : {results_path}')
results.head()

In [ ]:
evaluator = DetectionEvaluator(results, threshold=optimal_threshold)
evaluator.print_report()
evaluator.plot_all(save_dir=str(RESULTS_DIR))
evaluator.save_results_csv(save_dir=str(RESULTS_DIR))

metrics = evaluator.compute_metrics()
pd.Series(metrics)

In [ ]:
for filename in [
    'confusion_matrix.png',
    'roc_curve.png',
    'precision_recall_curve.png',
    'score_distribution.png',
    'metrics_over_time.png',
]:
    print(filename)
    show_saved_figure(filename)

In [ ]:
CONFIG['threshold_final'] = float(optimal_threshold)
save_config_and_metrics(CONFIG, metrics, str(RESULTS_DIR))

pd.Series(
    {
        'results_dir': str(RESULTS_DIR),
        'config_path': str(RESULTS_DIR / 'config.json'),
        'metrics_path': str(RESULTS_DIR / 'metrics.json'),
        'predictions_path': str(RESULTS_DIR / 'predictions.parquet'),
    }
)

## 4. Récapitulatif final

In [ ]:
final_recap = pd.Series(
    {
        'selected_model': CONFIG['selected_model']['model_name'],
        'window_seconds': CONFIG['selected_model']['window_seconds'],
        'n_components': CONFIG['selected_model']['n_components'],
        'min_flows': CONFIG['selected_model']['min_flows'],
        'threshold': metrics['Threshold'],
        'TP': metrics['TP'],
        'TN': metrics['TN'],
        'FP': metrics['FP'],
        'FN': metrics['FN'],
        'Precision': metrics['Precision'],
        'Recall': metrics['Recall'],
        'F1': metrics['F1'],
        'FPR': metrics['FPR'],
        'FNR': metrics['FNR'],
        'TNR': metrics['TNR'],
        'Accuracy': metrics['Accuracy'],
        'AUC_ROC': metrics['AUC_ROC'],
        'PR_AUC': metrics['PR_AUC'],
        'N_eval': metrics['N_eval'],
        'N_botnet': metrics['N_botnet'],
        'N_non_botnet': metrics['N_non_botnet'],
    },
    name='final_detection_pipeline',
)
display(final_recap.to_frame())